In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
from google.colab import files
uploaded = files.upload()

Saving Kryzotech RAG Assignmet.zip to Kryzotech RAG Assignmet (3).zip


In [11]:
import zipfile
import os

zip_path = "/content/Kryzotech RAG Assignmet.zip"
extract_path = "/content/Kryzotech_RAG_Assignment_Documents"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Done!")

Done!


In [12]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [13]:
import os

print(os.getcwd())
print(os.listdir("/content"))

/content
['.config', 'app.py', 'Kryzotech RAG Assignmet (3).zip', 'drive', 'Kryzotech_RAG_Assignment_Documents', 'Kryzotech RAG Assignmet (2).zip', '=0.46.1', 'Kryzotech RAG Assignmet (1).zip', 'Kryzotech RAG Assignmet.zip', 'sample_data']


In [14]:
# Install all required packages - this might take 2-3 minutes
!pip install bitsandbytes>=0.46.1 pypdf streamlit pyngrok langchain langchain-community langchain-core langchain-chroma langchain-text-splitters langchain-openai langchain-huggingface chromadb transformers torch accelerate sentence-transformers uvicorn fastapi -q

print("✅ All packages installed successfully!")

✅ All packages installed successfully!


## RAG with Streamlit front

In [15]:
%%writefile app.py

import langchain
import langchain_community
import langchain_core
import langchain_chroma
import langchain_text_splitters
import langchain_openai
import langchain_huggingface
import chromadb
import transformers
import torch
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline
import streamlit as st
import ast
import re



st.title("Hello RAG APP")
st.subheader("Be Aware with Streamlit")
st.text("Welcome to My First Interactive RAGChatbot")
st.sidebar.subheader("Chat History")

@st.cache_resource
def create_vector_store():
    loader = DirectoryLoader(
         path="/content/Kryzotech_RAG_Assignment_Documents/Kryzotech RAG Assignmet/papers",
         glob= "*.pdf",
         loader_cls=PyPDFLoader
)

    docs= loader.load()
    splittr = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20)

    split_docs =splittr.split_documents(docs)

    embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    )


    vectorstore = Chroma.from_documents(
       embedding=embedding,
       documents = split_docs
       )


    return vectorstore

#"What are the common diseases of Grape Leaves?"

@st.cache_resource
def LLM():
    model_id = "microsoft/Phi-3.5-mini-instruct"
    quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id,quantization_config=quantization_config,device_map="auto")
    pipeline = hf_pipeline(
        "text-generation",
        model = model,
        tokenizer = tokenizer,
        max_new_tokens = 200,
        temperature = 0.2,
        do_sample=False,
        return_full_text=False
      )
    llm = HuggingFacePipeline(pipeline=pipeline)
    return llm

vectorstore = create_vector_store()
retriver = vectorstore.as_retriever(
        search_type ="mmr",
        search_kwargs = {"k": 5, "lambda_mult": 0.5}
    )
llm = LLM()


query = st.text_input("Enter your query")

if "chat_history" not in st.session_state:
     st.session_state["chat_history"] = []

if st.button("Submit"):
    if not query.strip():
        st.warning("Please Enter your questions")

    else:
        with st.spinner("Searching documents and generating answer..."):
            result = retriver.invoke(query)
            context_text = "\n\n".join(doc.page_content for doc in result)
            prompt = PromptTemplate(
            template="""
            You are a helpful question-answer assistant for document based RAG System.

            Answer the user's question using ONLY the retrieved context provided below.

            If the context is insufficient, just say you don't know.

            Rules:
            1. Use only the information from the retrieved context.
            2. Give ONE answer only.
            3. Do NOT generate another question.
            4. Do NOT generate multiple question-answer pairs.
            5. Do NOT repeat the context.
            6. If the answer cannot be found in the context, say exactly: "I don't know."
            7. Keep the answer concise and directly relevant.
            8. Do not use your own knowledge or make assumptions.
            9. Return ONLY in this exact format:
               ["answer", "source", "page_number"]
            10. The source must be copied exactly from the retrieved context.
            11. The page_number must be copied exactly from the retrieved context.
            12. Do not generate additional questions or answers.
            13. Give only ONE concise answer.
            14. Do not invent or modify source names or page numbers.
            15. DO NOT write "Answer:","Source:","Page:",any explanation,markdown,code fences.

            Context:
            {context}

            Question:
            {question}

            Required format:
            ["answer", "source", "page_number"]
            Example:
            ["Rice blast causes lesions on leaves", "Rice Doctor, 2003; OU, 1985", "page 12"]
            """,
            input_variables=["context", "question"]
        )
            final_prompt = prompt.invoke({
            "context": context_text,
            "question": query
        })
            answer = llm.invoke(final_prompt)
            answer = str(answer).strip()
            raw_answer = answer
            if raw_answer.lower().startswith("answer"):
                raw_answer = raw_answer.split(":", 1)[1].strip()
            start = raw_answer.find("[")
            end = raw_answer.find("]")
            if start != -1 and end != -1:
                raw_answer = raw_answer[start:end+1]

            try:
                clean = ast.literal_eval(raw_answer)
                if isinstance(clean, list) and len(clean)>=3:
                    answer_text = clean[0]
                    source = clean[1]
                    page_number = clean[2]
                else:
                    answer_text = "NA"
                    source = "NA"
                    page_number = "NA"
            except(ValueError, SyntaxError):
                answer_text = raw_answer.strip()
                source = "NA"
                page_number = "NA"
            st.subheader("Answer is Given here")
            st.write(f"**Answer:** {answer_text}")
            st.write(f"**Source:** {source}")
            st.write(f"**Page:** {page_number}")

            st.session_state.chat_history.append({
            "question": query,
            "answer": answer_text,
            "source": source,
            "page": page_number
         })

            for chat in st.session_state.chat_history:
                st.sidebar.write(f"Question:{chat['question']}")
                st.sidebar.write(f"Answer:{chat['answer']}")
                st.sidebar.write(f"Source: {chat['source']}")
                st.sidebar.write(f"Page_No: {chat['page']}")



Overwriting app.py


In [16]:
from google.colab import userdata
NGROK_AUTH_TOKEN= userdata.get('NGROK_AUTH_TOKEN')





In [17]:
from pyngrok import ngrok
import subprocess
import time
import os

# Kill any existing processes
os.system("pkill -f streamlit")
os.system("pkill -f ngrok")
time.sleep(2)

# Set ngrok auth token
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Start Streamlit
print("Starting Streamlit app...")
process = subprocess.Popen([
    'streamlit', 'run', 'app.py',
    '--server.port', '8501',
    '--server.headless', 'true',
    '--server.enableCORS', 'false',
    '--server.enableXsrfProtection', 'false',
    '--server.maxUploadSize', '200'
])

# Wait for Streamlit to start
time.sleep(5)

# Create ngrok tunnel
print("Creating ngrok tunnel...")
public_url = ngrok.connect(8501)

print("\n" + "="*50)
print(f"Your app is running at: {public_url}")
print("="*50)


INFO:pyngrok.process:Updating authtoken for default "config_path" of "ngrok_path": /root/.config/ngrok/ngrok


Starting Streamlit app...


INFO:pyngrok.ngrok:Opening tunnel named: http-8501-5586e49f-3423-4fdf-be68-277484ad3b04
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:21+0000 lvl=info msg="no configuration paths supplied"
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:21+0000 lvl=info msg="using configuration at default config path" path=/root/.config/ngrok/ngrok.yml
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:21+0000 lvl=info msg="open config file" path=/root/.config/ngrok/ngrok.yml err=<nil>


Creating ngrok tunnel...


INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:22+0000 lvl=info msg="FIPS 140 mode" enabled=false
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:22+0000 lvl=info msg="starting web service" obj=web addr=127.0.0.1:4040 allow_hosts=[]
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:22+0000 lvl=info msg="client session established" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:22+0000 lvl=info msg="tunnel session started" obj=tunnels.session
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:23+0000 lvl=info msg=start pg=/api/tunnels id=5e3e5711b5785c55
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:23+0000 lvl=info msg=end pg=/api/tunnels id=5e3e5711b5785c55 status=200 dur=265.016µs
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:23+0000 lvl=info msg=start pg=/api/tunnels id=59a7682a0b2178f8
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:23+0000 lvl=info msg=end pg=/api/tunnels id=59a7682a0b2178f8 status=200 dur=1.024017ms
INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:23+0000 lvl=i


Your app is running at: NgrokTunnel: "https://punctuate-thinly-delicacy.ngrok-free.dev" -> "http://localhost:8501"


INFO:pyngrok.process.ngrok:t=2026-10-05T08:00:23+0000 lvl=info msg=end pg=/api/tunnels id=0266884094bda681 status=201 dur=380.241369ms


In [ ]:
What are the main causes of rice diseases?
What causes rice blast disease?
What are the major symptoms of bacterial leaf blight in rice?
Which pathogen is responsible for sheath blight disease?
What environmental conditions favor the development of rice blast?
How does bacterial leaf blight spread among rice plants?
What factors increase the severity of rice diseases?
What methods were used to control rice blast in the study?
Which rice variety showed the highest resistance to the disease?
What percentage of disease reduction was reported after treatment?

# RAG Model with FastAPI: POST /ask — takes a question, returns answer + sources
# GET /health — health check endpoint

# Proper request/response models , error handling (e.g., empty question, no documents indexed), and basic logging of each query + response time

In [18]:

import langchain
import langchain_community
import langchain_core
import langchain_chroma
import langchain_text_splitters
import langchain_openai
import langchain_huggingface
import chromadb
import transformers
import torch
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline
import streamlit as st
import ast
from fastapi import FastAPI
from pydantic import BaseModel
import time
import logging



loader = DirectoryLoader(
         path="/content/Kryzotech_RAG_Assignment_Documents/Kryzotech RAG Assignmet/papers",
         glob= "*.pdf",
         loader_cls=PyPDFLoader)

docs= loader.load()
splittr = RecursiveCharacterTextSplitter(
chunk_size = 500,
chunk_overlap = 20)

split_docs =splittr.split_documents(docs)

embedding = HuggingFaceEmbeddings(
model_name = "sentence-transformers/all-MiniLM-L6-v2"
)


vectorstore = Chroma.from_documents(
   embedding=embedding,
   documents = split_docs,
   persist_directory="/content/drive/MyDrive/Kryzotech_RAG/chroma4_db"
   )




def LLM():
    model_id = "microsoft/Phi-3.5-mini-instruct"
    quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id,quantization_config=quantization_config,device_map="auto")
    pipeline = hf_pipeline(
        "text-generation",
        model = model,
        tokenizer = tokenizer,
        max_new_tokens = 200,
        temperature = 0.2,
        do_sample=False,
        return_full_text=False
      )
    llm = HuggingFacePipeline(pipeline=pipeline)
    return llm


retriver = vectorstore.as_retriever(
        search_type ="mmr",
        search_kwargs = {"k": 5, "lambda_mult": 0.5}
    )

llm = LLM()




INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22952a9b81c0bfd99800fabf352f69c7ccd46c5e43%22 "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json?%2Fsentence-transformers%2Fall-MiniLM-L6-v2%2Fresolve%2Fmain%2Fconfig_sentence_transformers.js

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/tokenizer_config.json?%2Fsentence-transformers%2Fall-MiniL

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/microsoft/Phi-3.5-mini-instruct/resolve/main/generation_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/Phi-3.5-mini-instruct/2fe192450127e6a83f7441aef6e3ca586c338b77/generation_config.json?%2Fmicrosoft%2FPhi-3.5-mini-instruct%2Fresolve%2Fmain%2Fgeneration_config.json=&etag=%2293dfd6366eab0bbae1613c5deab3a43bc989e5f8%22 "HTTP/1.1 200 OK"


In [19]:

import langchain
import langchain_community
import langchain_core
import langchain_chroma
import langchain_text_splitters
import langchain_openai
import langchain_huggingface
import chromadb
import transformers
import torch
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline
import streamlit as st
import ast
from fastapi import FastAPI,status,HTTPException
from pydantic import BaseModel
import time
import logging

logging.basicConfig(level=logging.INFO,force=True)
app = FastAPI()


class User(BaseModel):
    question: str

class QueryResponse(BaseModel):
    question: str
    answer: str
    source: str
    page_no: str


class HealthResponse(BaseModel):
  status: str

@app.get('/health',response_model=HealthResponse)
def healthcheak():
    return{"status": "healthy"}

@app.post('/ask',response_model=QueryResponse)
def ask_question(request: User):
    query = request.question
    start_time = time.time()
    if not request.question.strip():
      raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST,detail="Question can't be empty")
    result = retriver.invoke(query)
    context_text = "\n\n".join(
            doc.page_content for doc in result
        )

    prompt = PromptTemplate(
            template="""
            You are a helpful question-answer assistant for document based RAG System.

            Answer the user's question using ONLY the retrieved context provided below.

            If the context is insufficient, just say you don't know.

            Rules:
            1. Use only the information from the retrieved context.
            2. Give ONE answer only.
            3. Do NOT generate another question.
            4. Do NOT generate multiple question-answer pairs.
            5. Do NOT repeat the context.
            6. If the answer cannot be found in the context, say exactly: "I don't know."
            7. Keep the answer concise and directly relevant.
            8. Do not use your own knowledge or make assumptions.
            9. Return ONLY in this exact format:
               ["answer", "source", "page_number"]
            10. The source must be copied exactly from the retrieved context.
            11. The page_number must be copied exactly from the retrieved context.
            12. Do not generate additional questions or answers.
            13. Give only ONE concise answer.
            14. Do not invent or modify source names or page numbers.
            15. DO NOT write "Answer:","Source:","Page:",any explanation,markdown,code fences.

            Context:
            {context}

            Question:
            {question}

            Required format:
            ["answer", "source", "page_number"]
            Example:
            ["Rice blast causes lesions on leaves", "Rice Doctor, 2003; OU, 1985", "page 12"]
            """,
            input_variables=["context", "question"]
        )
    final_prompt = prompt.invoke({
            "context": context_text,
            "question": query
        })


    answer = llm.invoke(final_prompt)

    try:
        answer = str(answer).strip()
        raw_answer = answer
        if raw_answer.lower().startswith("answer"):
            raw_answer = raw_answer.split(":", 1)[1].strip()
            clean = ast.literal_eval(raw_answer.strip())

        if isinstance(clean, list) and len(clean)>=3:
                answer_text = clean[0]
                source = clean[1]
                page_number = clean[2]
        else:
            answer_text = "NA"
            source = "NA"
            page_number = "NA"

    except(ValueError, SyntaxError):
        answer_text = answer.strip()
        source = "NA"
        page_number = "NA"

    response_time= time.time()-start_time
    logging.info(
    f"Query: {query} | "
    f"Response_Time: {response_time:.2f} seconds"
)


    return {
        "question": query,
        "answer": answer_text,
        "source": source,
        "page_no":page_number

    }



In [23]:
!pkill -f uvicorn

In [24]:
for route in app.routes:
    print(route.path, route.methods)

/openapi.json {'HEAD', 'GET'}
/docs {'HEAD', 'GET'}
/docs/oauth2-redirect {'HEAD', 'GET'}
/redoc {'HEAD', 'GET'}
/health {'GET'}
/ask {'POST'}


In [25]:
import uvicorn
import threading

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

INFO:     Started server process [32739]
INFO:     Waiting for application startup.


In [26]:
!curl http://127.0.0.1:8000/ask

INFO:     127.0.0.1:55154 - "GET /ask HTTP/1.1" 405 Method Not Allowed
{"detail":"Method Not Allowed"}

In [27]:
!npx localtunnel --port 8000

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹your url is: https://fast-melons-bow.loca.lt
INFO:     103.165.186.12:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     103.165.186.12:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     103.165.186.12:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     103.165.186.12:0 - "GET /openapi.json HTTP/1.1" 200 OK


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
INFO:root:Query: What environmental conditions favor the development of rice blast?|Answer: Sunny time of the day|Response_Time: 8.83seconds


INFO:     103.165.186.12:0 - "POST /ask HTTP/1.1" 200 OK
^C


# POST /upload — allows uploading a new PDF to add to the index


In [ ]:
%%writefile /content/upload.py

import langchain
import langchain_community
import langchain_core
import langchain_chroma
import langchain_text_splitters
import langchain_openai
import langchain_huggingface
import chromadb
import transformers
import torch
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline
import ast
from fastapi import FastAPI,UploadFile,File
from pydantic import BaseModel
import time
import logging
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()
def create_vector_store(path:str):
    loader = DirectoryLoader(
         path=path,
         glob= "*.pdf",
         loader_cls=PyPDFLoader
)

    docs= loader.load()
    splittr = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20)

    split_docs =splittr.split_documents(docs)

    embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    )


    vectorstore = Chroma.from_documents(
       embedding=embedding,
       documents = split_docs,
       persist_directory="/content/drive/MyDrive/Kryzotech_RAG/chroma_db"
       )


    return vectorstore

def LLM():
    model_id = "microsoft/Phi-3.5-mini-instruct"
    quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id,quantization_config=quantization_config,device_map="auto")
    pipeline = hf_pipeline(
        "text-generation",
        model = model,
        tokenizer = tokenizer,
        max_new_tokens = 200,
        temperature = 0.2,
        do_sample=False,
        return_full_text=False
      )
    llm = HuggingFacePipeline(pipeline=pipeline)
    return llm



class UploadResponse(BaseModel):
    message: str
    filename: str
class User(BaseModel):
    question: str

class QueryResponse(BaseModel):
    question: str
    answer: str
    source: str
    page_no: str



@app.get('/')
def home():
    return{
        "message": "RAG MODEL IS Loading"
    }
@app.post('/upload',response_model=UploadResponse)
async def upload_pdf(file: UploadFile= File()):
    paper_dirs = "/content/Upload_Papers"
    os.makedirs(paper_dirs,exist_ok=True)
    filepath = os.path.join(paper_dirs,file.filename)
    with open(filepath,"wb") as f:
        f.write(await file.read())

    return {
        "message": "Pdf Uploaded Successfully",
        "filename": file.filename
    }



Overwriting /content/upload.py


In [ ]:
!pkill -f uvicorn

In [ ]:
!uvicorn upload:app --host 0.0.0.0 --port 8001 > fastapi.log 2>&1 &

In [ ]:
!cat fastapi.log

In [ ]:
!npx localtunnel --port 8001

⠙⠹⠸⠼⠴⠦your url is: https://dull-banks-cough.loca.lt
^C


# POST /ask — takes a question, returns answer + sources from the uploaded pdf's


In [ ]:
%%writefile /content/upload_ask.py

import langchain
import langchain_community
import langchain_core
import langchain_chroma
import langchain_text_splitters
import langchain_openai
import langchain_huggingface
import chromadb
import transformers
import torch
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import BitsAndBytesConfig
from transformers import pipeline as hf_pipeline
from langchain_huggingface import HuggingFacePipeline
import ast
from fastapi import FastAPI,UploadFile,File
from pydantic import BaseModel
import time
import logging
import os

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()
def create_vector_store(path:str):
    loader = DirectoryLoader(
         path=path,
         glob= "*.pdf",
         loader_cls=PyPDFLoader
)

    docs= loader.load()
    splittr = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 20)

    split_docs =splittr.split_documents(docs)

    embedding = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
    )


    vectorstore = Chroma.from_documents(
       embedding=embedding,
       documents = split_docs,
       persist_directory="/content/drive/MyDrive/Kryzotech_RAG/chroma_db"
       )


    return vectorstore

def LLM():
    model_id = "microsoft/Phi-3.5-mini-instruct"
    quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id,quantization_config=quantization_config,device_map="auto")
    pipeline = hf_pipeline(
        "text-generation",
        model = model,
        tokenizer = tokenizer,
        max_new_tokens = 200,
        temperature = 0.2,
        do_sample=False,
        return_full_text=False
      )
    llm = HuggingFacePipeline(pipeline=pipeline)
    return llm



class UploadResponse(BaseModel):
    message: str
    filename: str
class User(BaseModel):
    question: str

class QueryResponse(BaseModel):
    question: str
    answer: str
    source: str
    page_no: str



@app.get('/')
def home():
    return{
        "message": "RAG MODEL IS Loading"
    }
@app.post('/upload',response_model=UploadResponse)
async def upload_pdf(file: UploadFile= File()):
    paper_dirs = "/content/Upload_Papers"
    os.makedirs(paper_dirs,exist_ok=True)
    filepath = os.path.join(paper_dirs,file.filename)
    with open(filepath,"wb") as f:
        f.write(await file.read())

    return {
        "message": "Pdf Uploaded Successfully",
        "filename": file.filename
    }



vectorstore = create_vector_store("/content/Upload_Papers")
retriver = vectorstore.as_retriever(
        search_type ="mmr",
        search_kwargs = {"k": 5, "lambda_mult": 0.5}

    )
llm = LLM()

@app.post('/ask',response_model=QueryResponse)
def ask_question(request: User):

    query = request.question
    result = retriver.invoke(query)
    context_text = "\n\n".join(
            doc.page_content for doc in result
        )

    prompt = PromptTemplate(
            template="""
            You are a helpful question-answer assistant for document based RAG System.

            Answer the user's question using ONLY the retrieved context provided below.

            If the context is insufficient, just say you don't know.

            Rules:
            1. Use only the information from the retrieved context.
            2. Give ONE answer only.
            3. Do NOT generate another question.
            4. Do NOT generate multiple question-answer pairs.
            5. Do NOT repeat the context.
            6. If the answer cannot be found in the context, say exactly: "I don't know."
            7. Keep the answer concise and directly relevant.
            8. Do not use your own knowledge or make assumptions.
            9. Return ONLY in this exact format:
               ["answer", "source", "page_number"]
            10. The source must be copied exactly from the retrieved context.
            11. The page_number must be copied exactly from the retrieved context.
            12. Do not generate additional questions or answers.
            13. Give only ONE concise answer.
            14. Do not invent or modify source names or page numbers.
            15. DO NOT write "Answer:","Source:","Page:",any explanation,markdown,code fences.

            Context:
            {context}

            Question:
            {question}

            Required format:
            ["answer", "source", "page_number"]
            Example:
            ["Rice blast causes lesions on leaves", "Rice Doctor, 2003; OU, 1985", "page 12"]
            """,
            input_variables=["context", "question"]
        )
    final_prompt = prompt.invoke({
            "context": context_text,
            "question": query
        })


    answer = llm.invoke(final_prompt)
    try:
        answer = str(answer).strip()
        raw_answer = answer
        if raw_answer.lower().startswith("answer"):
            raw_answer = raw_answer.split(":", 1)[1].strip()
            clean = ast.literal_eval(raw_answer.strip())

        if isinstance(clean, list) and len(clean)>=3:
                answer_text = clean[0]
                source = clean[1]
                page_number = clean[2]
        else:
            answer_text = "NA"
            source = "NA"
            page_number = "NA"

    except(ValueError, SyntaxError):
        answer_text = answer.strip()
        source = "NA"
        page_number = "NA"

    return {
        "question": query,
        "answer": answer_text,
        "source": source,
        "page_no":page_number
    }


Writing /content/upload_ask.py


In [ ]:
!pkill -f uvicorn

In [ ]:
!uvicorn upload_ask:app --host 0.0.0.0 --port 8002 > fastapi.log 2>&1 &

In [ ]:
!cat fastapi.log

In [ ]:
!npx localtunnel --port 8002

⠙⠹⠸⠼⠴⠦⠧⠇⠏your url is: https://floppy-teams-lick.loca.lt
^C


# 1.GET /health — health check endpoint


# Didn't understand how to execute these 2:

# 2.Proper request/response models , error handling (e.g., empty question, no documents indexed), and basic logging of each query + response time
